# CIBC — Digital Insurance Platform Acquisition
## Module 4 — Quantitative Risk Assessment: Indicators & Warnings (I&W) Analysis
### ALY6130 | Group 7 | June 2026

---

This notebook applies quantitative risk analysis to the three highest-priority risks identified in the Module 3 Risk Assessment:

| Risk | Description | L | I | Score | Priority |
|------|-------------|---|---|-------|----------|
| R1 | Data Privacy Breach | 9 | 8 | 72 | HIGH |
| R2 | Regulatory & Compliance Exposure | 7 | 8 | 56 | HIGH |
| R3 | First-Mover Market Capture (Positive) | 7 | 8 | 56 | HIGH |

**Quantitative methods applied:**
- R1: Monte Carlo Simulation (triangular distribution, 10,000 iterations)
- R2: Sensitivity Analysis (Tornado Chart)
- R3: Expected Monetary Value (EMV) across adoption scenarios

**I&W Framework:** Each risk is assessed through the Indicators & Warnings lens — identifying observable leading indicators and defining warning thresholds that signal escalating risk before a crisis occurs.

---
**Source:** Group 7 Risk Register and Risk Calculation Sheet, ALY6130, Spring 2026.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SETUP — Import libraries
# ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import triang
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)   # reproducibility
N = 10000            # Monte Carlo iterations

print('Libraries loaded. Monte Carlo iterations:', N)

---
## PART 1 — Risk Register: Three Risks Under Analysis

The I&W analysis technique identifies **Indicators** — observable, measurable signals that a risk is developing — and **Warnings** — threshold breaches that require a defined management response.

This section documents the I&W framework for each risk before quantitative modeling is applied.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Risk Register — Three risks under analysis
# Scores from Module 3 final submission (Module3_Group7_Final_APPROVED_VERSION)
# ─────────────────────────────────────────────────────────────────

risks = pd.DataFrame({
    'Risk #':    [1, 2, 3],
    'Risk Name': [
        'Data Privacy Breach',
        'Regulatory & Compliance Exposure',
        'First-Mover Market Capture'
    ],
    'Type':      ['Negative', 'Negative', 'Positive'],
    'Likelihood':[9, 7, 7],
    'Impact':    [8, 8, 8],
    'Risk Score':[72, 56, 56],
    'Priority':  ['H', 'H', 'H'],
    'Quant Method': [
        'Monte Carlo Simulation',
        'Sensitivity (Tornado) Analysis',
        'Expected Monetary Value (EMV)'
    ],
    'Risk Owner': [
        'CISO / Chief Compliance Officer',
        'Chief Compliance Officer',
        'Chief Strategy Officer / CFO'
    ]
})

print('Three risks under quantitative analysis:\n')
print(risks[['Risk #','Risk Name','Type','Likelihood','Impact','Risk Score','Priority','Quant Method']]
      .to_string(index=False))

---
## PART 2 — I&W Framework: Indicators and Warning Thresholds

The I&W technique maps each risk to observable leading indicators and defines Green / Amber / Red warning thresholds.
These thresholds were developed in Module 3 as KRIs and are now formalized as quantitative warning boundaries.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# I&W FRAMEWORK — Indicators and Warning Thresholds for all three risks
# Source: KRI Framework, Group 7 Risk Register, ALY6130 Spring 2026
# ─────────────────────────────────────────────────────────────────

iw_framework = [
    {
        'Risk': 'R1 — Data Privacy Breach',
        'Primary Indicator': 'No. of critical security vulnerabilities unpatched >72 hours',
        'Secondary Indicators': [
            'SOC alert volume (anomalous access attempts per 24h)',
            'Integration security scan pass rate (%)',
            'Penetration test findings unresolved >14 days'
        ],
        'GREEN  (Within Appetite)': 'Zero critical vulnerabilities unpatched >72 hrs. SOC alerts within baseline.',
        'AMBER  (Warning)':         '1 critical vulnerability unpatched >72 hrs OR SOC alerts 2x baseline. CISO notified. Weekly reporting begins.',
        'RED    (Appetite Breach)': '2+ unpatched critical vulnerabilities >72 hrs OR any confirmed unauthorised access. CEO/Board escalation. Integration paused.',
        'Warning Owner': 'CISO'
    },
    {
        'Risk': 'R2 — Regulatory & Compliance Exposure',
        'Primary Indicator': 'No. of open compliance findings or unimplemented regulatory changes older than 30 days',
        'Secondary Indicators': [
            'Number of provinces with confirmed insurance distribution licence',
            'Days to Bill C-27 compliance readiness',
            'Number of outstanding OSFI Guideline B-10 vendor exit strategies'
        ],
        'GREEN  (Within Appetite)': 'Zero open findings >30 days. All provincial licences on schedule.',
        'AMBER  (Warning)':         '1-2 open findings >30 days. Compliance committee convened. Monthly board reporting begins.',
        'RED    (Appetite Breach)': '3+ findings unresolved OR formal regulatory inquiry received. CEO/Board escalation. Platform sales halted in affected provinces.',
        'Warning Owner': 'Chief Compliance Officer'
    },
    {
        'Risk': 'R3 — First-Mover Market Capture (Positive)',
        'Primary Indicator': 'Monthly digital insurance client adoption rate (% of active CIBC digital banking clients)',
        'Secondary Indicators': [
            'Confirmed launch date from TD Bank or RBC for integrated insurance product',
            '% of CIBC advisors licensed and trained for insurance cross-sell',
            'Monthly new insurance policy applications'
        ],
        'GREEN  (Within Appetite)': 'Adoption >5% at 12 months. No confirmed competitor launch. Advisor licensing >80%.',
        'AMBER  (Warning)':         'Adoption 2-5% at 12 months OR TD/RBC announces firm go-live date within 6 months. Strategy review activated.',
        'RED    (Appetite Breach)': 'Adoption <2% at 12 months OR TD/RBC launches before CIBC achieves meaningful share. Immediate CEO/Board review.',
        'Warning Owner': 'Chief Strategy Officer'
    }
]

for r in iw_framework:
    print('=' * 70)
    print(f"RISK: {r['Risk']}")
    print(f"  Primary Indicator:  {r['Primary Indicator']}")
    print(f"  Secondary Indicators:")
    for si in r['Secondary Indicators']:
        print(f"    • {si}")
    print(f"  GREEN:  {r['GREEN  (Within Appetite)']}")
    print(f"  AMBER:  {r['AMBER  (Warning)']}")
    print(f"  RED:    {r['RED    (Appetite Breach)']}")
    print(f"  Owner:  {r['Warning Owner']}")
    print()

---
## PART 3 — R1: Monte Carlo Simulation (Data Privacy Breach)

**Method:** Triangular distribution modeling total financial loss across 10,000 simulated scenarios.

**Cost inputs from CIBC risk register:**
- Optimistic: CAD $25M (PIPEDA fine floor + minimal class-action)
- Most Likely: CAD $75M (mid-range fine + class-action + remediation)
- Pessimistic: CAD $150M (maximum fine + full class-action + 6–12 month shutdown)

**I&W Connection:** The Monte Carlo output quantifies the financial consequence distribution that the I&W warning thresholds are designed to prevent.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# R1 — MONTE CARLO SIMULATION
# Triangular distribution: Optimistic=$25M, MostLikely=$75M, Pessimistic=$150M
# Cost inputs sourced directly from Group 7 risk register R1 consequence field
# ─────────────────────────────────────────────────────────────────

r1_opt  = 25    # CAD $M — optimistic (PIPEDA fine floor)
r1_ml   = 75    # CAD $M — most likely (mid-range scenario)
r1_pess = 150   # CAD $M — pessimistic (full class-action + shutdown)

# Triangular distribution shape parameter
c = (r1_ml - r1_opt) / (r1_pess - r1_opt)

# Run simulation
r1_losses = triang.rvs(c, loc=r1_opt, scale=r1_pess - r1_opt, size=N)

# Summary statistics
r1_mean  = np.mean(r1_losses)
r1_p50   = np.percentile(r1_losses, 50)
r1_p90   = np.percentile(r1_losses, 90)
r1_p95   = np.percentile(r1_losses, 95)
r1_p99   = np.percentile(r1_losses, 99)
r1_prob_over_100 = np.mean(r1_losses > 100) * 100
r1_prob_over_125 = np.mean(r1_losses > 125) * 100

print('R1 — Data Privacy Breach: Monte Carlo Results')
print('=' * 50)
print(f'  Simulation runs:          {N:,}')
print(f'  Input — Optimistic:       CAD ${r1_opt}M')
print(f'  Input — Most Likely:      CAD ${r1_ml}M')
print(f'  Input — Pessimistic:      CAD ${r1_pess}M')
print(f'  Mean expected loss:       CAD ${r1_mean:.1f}M')
print(f'  Median (50th pct):        CAD ${r1_p50:.1f}M')
print(f'  90th percentile:          CAD ${r1_p90:.1f}M')
print(f'  95th percentile:          CAD ${r1_p95:.1f}M')
print(f'  99th percentile:          CAD ${r1_p99:.1f}M')
print(f'  P(Loss > CAD $100M):      {r1_prob_over_100:.1f}%')
print(f'  P(Loss > CAD $125M):      {r1_prob_over_125:.1f}%')
print()
print(f'  >> Recommended contingency reserve: CAD ${r1_p90:.0f}M (90th percentile)')
print(f'  >> Board should stress-test to:      CAD ${r1_p95:.0f}M (95th percentile)')

In [ ]:
# ─── R1 CHART ───────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(11, 5))
ax1.hist(r1_losses, bins=80, color='#D42020', alpha=0.75, edgecolor='#8B0000', linewidth=0.4)
ax1.axvline(r1_mean, color='#1F3864', lw=2.0, ls='--', label=f'Mean = CAD ${r1_mean:.1f}M')
ax1.axvline(r1_p90,  color='#FF6600', lw=2.0, ls='-',  label=f'90th Pct = CAD ${r1_p90:.1f}M  ← Recommended Reserve')
ax1.axvline(r1_p95,  color='#8B0000', lw=2.0, ls=':',  label=f'95th Pct = CAD ${r1_p95:.1f}M  ← Board Stress Test')
ax1.set_xlabel('Total Financial Loss (CAD $M)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency (out of 10,000 simulations)', fontsize=11, fontweight='bold')
ax1.set_title('R1 — Data Privacy Breach\nMonte Carlo Simulation: Total Financial Loss Distribution (10,000 iterations)',
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.text(0.97, 0.70,
    f'Optimistic:  CAD ${r1_opt}M\nMost Likely: CAD ${r1_ml}M\nPessimistic: CAD ${r1_pess}M\n'
    f'P(Loss > $100M): {r1_prob_over_100:.1f}%',
    transform=ax1.transAxes, ha='right', va='top', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='#FFF3F3', edgecolor='#D42020', alpha=0.9))
fig1.text(0.01, -0.04,
    'Figure 1. Monte Carlo simulation — R1 Data Privacy Breach. '
    'Triangular distribution inputs: Optimistic=CAD $25M (PIPEDA fine floor), '
    'Most Likely=CAD $75M (mid-range fine + class-action + remediation), '
    'Pessimistic=CAD $150M (maximum fine + full class-action + 6-12 month shutdown). '
    'Source: Group 7 Risk Register, ALY6130, Spring 2026.',
    fontsize=7, color='#555555', style='italic')
fig1.subplots_adjust(bottom=0.15)
plt.savefig('fig1_r1_montecarlo.png', dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 1 saved: fig1_r1_montecarlo.png')

---
## PART 4 — R2: Sensitivity (Tornado) Analysis (Regulatory & Compliance Exposure)

**Method:** Each key impact driver is varied ±30% independently while all others are held at base value. The resulting swing in total impact is plotted as a tornado chart, ranked from largest to smallest swing.

**Base impact:** CAD $56M (derived from L=7 × I=8 risk score, consistent with Module 3 register).

**I&W Connection:** The sensitivity output identifies which indicator CIBC should monitor most intensively — the variable with the widest swing is the one whose early-warning indicator carries the greatest consequence.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# R2 — SENSITIVITY ANALYSIS (TORNADO CHART)
# Base impact = CAD $56M (L=7 x I=8)
# Five key drivers varied ±30% independently
# ─────────────────────────────────────────────────────────────────

base_impact = 56.0  # CAD $M

# (Variable label, sensitivity weight, upside %, downside %)
# Weight = proportion of base impact attributable to this variable
variables = [
    ('Likelihood of Regulatory Halt',    0.70, +30, -30),
    ('Regulatory Fine Severity',         0.55, +25, -25),
    ('Provincial Launch Delay Duration', 0.45, +20, -20),
    ('No. of Provinces Affected',        0.35, +15, -15),
    ('Legal & Remediation Costs',        0.25, +12, -10),
]

print('R2 — Regulatory & Compliance Exposure: Sensitivity Analysis')
print('=' * 65)
print(f'  Base total impact: CAD ${base_impact}M')
print()
print(f'{"Variable":<40} {"Low (−30%)":>12} {"Base":>10} {"High (+30%)":>12}')
print('-' * 76)
for label, weight, up_pct, down_pct in variables:
    high_val = base_impact * (1 + weight * up_pct / 100)
    low_val  = base_impact * (1 + weight * down_pct / 100)
    swing    = high_val - low_val
    print(f'{label:<40} CAD ${low_val:>6.1f}M  CAD ${base_impact:>5.1f}M  CAD ${high_val:>6.1f}M  (swing: ${swing:.1f}M)')
print()
print('  >> Highest-impact driver: Likelihood of Regulatory Halt')
print('     CIBC should prioritise monitoring open compliance findings and')
print('     provincial licensing status — these directly determine halt probability.')

In [ ]:
# ─── R2 CHART ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(11, 5))
y_positions = range(len(variables))
for i, (label, weight, up_pct, down_pct) in enumerate(variables):
    high_val = base_impact * (1 + weight * up_pct / 100)
    low_val  = base_impact * (1 + weight * down_pct / 100)
    ax2.barh(i, high_val - base_impact, left=base_impact,
             color='#D42020', alpha=0.82, height=0.55)
    ax2.barh(i, low_val  - base_impact, left=base_impact,
             color='#4472C4', alpha=0.82, height=0.55)
    ax2.text(high_val + 0.4, i, f'+{up_pct}%',
             va='center', fontsize=9, color='#8B0000', fontweight='bold')
    ax2.text(low_val  - 0.4, i, f'{down_pct}%',
             va='center', fontsize=9, color='#1F3864', fontweight='bold', ha='right')

ax2.set_yticks(list(y_positions))
ax2.set_yticklabels([v[0] for v in variables], fontsize=10)
ax2.axvline(base_impact, color='black', lw=1.5, ls='--',
            label=f'Base Impact = CAD ${base_impact}M')
ax2.set_xlabel('Total Financial Impact (CAD $M)', fontsize=11, fontweight='bold')
ax2.set_title('R2 — Regulatory & Compliance Exposure\nSensitivity (Tornado) Analysis — Key Impact Drivers',
              fontsize=12, fontweight='bold')
red_p  = mpatches.Patch(color='#D42020', alpha=0.82, label='Upside — variable increases 30%')
blue_p = mpatches.Patch(color='#4472C4', alpha=0.82, label='Downside — variable decreases 30%')
ax2.legend(handles=[red_p, blue_p], fontsize=9, loc='lower right')
fig2.text(0.01, -0.04,
    'Figure 2. Sensitivity (tornado) analysis — R2 Regulatory & Compliance Exposure. '
    'Base impact CAD $56M from L=7 × I=8. Each variable varied ±30% independently. '
    'Wider bars = greater sensitivity = higher monitoring priority. '
    'Source: Group 7 Risk Register, ALY6130, Spring 2026.',
    fontsize=7, color='#555555', style='italic')
fig2.subplots_adjust(bottom=0.15)
plt.savefig('fig2_r2_sensitivity.png', dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 2 saved: fig2_r2_sensitivity.png')

---
## PART 5 — R3: Expected Monetary Value (First-Mover Market Capture)

**Method:** EMV = Probability × Net Value, summed across four adoption scenarios.

**Revenue inputs from CIBC risk register:**
- Annual premiums at 5% adoption = CAD $165M (from register)
- Cross-sell revenue uplift = CAD $50–100M (McKinsey 2023)
- Opportunity cost if missed = CAD $40–60M additional spend (from register)

**I&W Connection:** The primary I&W indicator for R3 is the monthly adoption rate — the EMV output quantifies exactly what is at stake if each warning threshold is breached.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# R3 — EXPECTED MONETARY VALUE (EMV)
# Four adoption scenarios with probability-weighted net value
# ─────────────────────────────────────────────────────────────────

scenarios = {
    'Strong First-Mover (5% adoption)':      {'prob': 0.70, 'premium': 165, 'crosssell': 75,  'opp_cost': 0},
    'Partial First-Mover (3% adoption)':     {'prob': 0.15, 'premium': 99,  'crosssell': 40,  'opp_cost': 20},
    'Late Entry (competitor launches first)':{'prob': 0.10, 'premium': 60,  'crosssell': 20,  'opp_cost': 50},
    'Failed Launch (<1% adoption)':          {'prob': 0.05, 'premium': 20,  'crosssell': 5,   'opp_cost': 60},
}

# Verify probabilities sum to 1.0
assert abs(sum(s['prob'] for s in scenarios.values()) - 1.0) < 0.001, 'Probabilities must sum to 1.0'

print('R3 — First-Mover Market Capture: EMV Analysis')
print('=' * 80)
print(f'{"Scenario":<45} {"Prob":>6} {"Premium":>10} {"X-Sell":>8} {"OppCost":>9} {"Net":>8} {"EMV":>10}')
print('-' * 80)

total_emv = 0
scenario_emvs = {}
for label, s in scenarios.items():
    net = s['premium'] + s['crosssell'] - s['opp_cost']
    emv = s['prob'] * net
    total_emv += emv
    scenario_emvs[label] = emv
    print(f"{label:<45} {s['prob']:>5.0%}  ${s['premium']:>7}M  ${s['crosssell']:>5}M  ${s['opp_cost']:>6}M  ${net:>5}M  ${emv:>7.1f}M")

print('=' * 80)
print(f'{"TOTAL EMV (Risk R3 — First-Mover Market Capture)":<62} CAD ${total_emv:.1f}M')
print()
print(f'  >> Total EMV of CAD ${total_emv:.1f}M strongly supports pursuing this opportunity.')
print(f'  >> The Strong First-Mover scenario (70% probability) contributes ${0.70*(165+75):.1f}M alone.')
print(f'  >> RED warning (adoption <2%) represents estimated CAD ${0.05*(20+5-60):.1f}M net loss scenario.')

In [ ]:
# ─── R3 CHART ───────────────────────────────────────────────────
labels   = list(scenario_emvs.keys())
emv_vals = list(scenario_emvs.values())
probs    = [s['prob']     for s in scenarios.values()]
premiums = [s['premium']  for s in scenarios.values()]
crossells= [s['crosssell']for s in scenarios.values()]
oppcosts = [s['opp_cost'] for s in scenarios.values()]
short_labels = [
    'Strong\nFirst-Mover\n(5% adoption)',
    'Partial\nFirst-Mover\n(3% adoption)',
    'Late Entry\n(competitor\nlaunches first)',
    'Failed\nLaunch\n(<1% adoption)'
]

colors = ['#2E7D32', '#66BB6A', '#FF6600', '#D42020']
fig3, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(short_labels, emv_vals, color=colors, alpha=0.85,
                   edgecolor='#333333', linewidth=0.8, width=0.55)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_ylabel('EMV Contribution (CAD $M)', fontsize=10, fontweight='bold')
axes[0].set_title(f'R3 — EMV by Scenario\nTotal EMV = CAD ${total_emv:.1f}M', fontsize=11, fontweight='bold')
for bar, val in zip(bars, emv_vals):
    ypos = bar.get_height() + 1 if val >= 0 else bar.get_height() - 4
    axes[0].text(bar.get_x() + bar.get_width()/2, ypos,
                f'${val:.1f}M', ha='center', fontsize=9, fontweight='bold')

x = np.arange(len(short_labels))
w = 0.26
axes[1].bar(x - w, premiums,  width=w, color='#1B5E20', alpha=0.85, label='Premium Revenue')
axes[1].bar(x,     crossells, width=w, color='#0D47A1', alpha=0.85, label='Cross-Sell Uplift')
axes[1].bar(x + w, oppcosts,  width=w, color='#B71C1C', alpha=0.85, label='Opportunity Cost')
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_labels, fontsize=8.5)
axes[1].set_ylabel('CAD $M', fontsize=10, fontweight='bold')
axes[1].set_title('R3 — Revenue & Cost Breakdown\nby Adoption Scenario', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)

fig3.text(0.01, -0.04,
    'Figure 3. EMV analysis — R3 First-Mover Market Capture. '
    'Premium inputs: 5% adoption of 11M clients. Cross-sell uplift per McKinsey (2023). '
    'Opportunity cost per project document CAD $40-60M. EMV = Probability × Net Value. '
    'Source: Group 7 Risk Register, ALY6130, Spring 2026.',
    fontsize=7, color='#555555', style='italic')
fig3.tight_layout()
fig3.subplots_adjust(bottom=0.15)
plt.savefig('fig3_r3_emv.png', dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 3 saved: fig3_r3_emv.png')

---
## PART 6 — Quantitative Results Summary & RT&RP Update

This section summarizes all quantitative outputs and documents the updated Risk Treatment & Response Plan entries for R1, R2, and R3.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SUMMARY TABLE — Quantitative Results
# ─────────────────────────────────────────────────────────────────

summary = pd.DataFrame({
    'Risk': ['R1 — Data Privacy Breach',
             'R2 — Regulatory Exposure',
             'R3 — First-Mover Capture'],
    'Quant Method': [
        'Monte Carlo (Triangular, N=10,000)',
        'Sensitivity / Tornado Analysis',
        'Expected Monetary Value (EMV)'
    ],
    'Key Output': [
        f'Mean loss CAD ${r1_mean:.0f}M | 90th pct CAD ${r1_p90:.0f}M | P(>$100M)={r1_prob_over_100:.1f}%',
        'Top driver: Likelihood of Regulatory Halt (highest swing). Monitor compliance findings KRI.',
        f'Total EMV = CAD ${total_emv:.1f}M. Strong First-Mover scenario drives 90% of value.'
    ],
    'RT&RP Action': [
        f'Set contingency reserve at CAD ${r1_p90:.0f}M (90th pct). Board stress-test at CAD ${r1_p95:.0f}M.',
        'Prioritise compliance findings KRI. Any halt probability increase triggers immediate board review.',
        f'Pursue opportunity actively. CAD ${total_emv:.0f}M EMV justifies full investment. Track adoption monthly.'
    ]
})

print('QUANTITATIVE RISK ASSESSMENT — SUMMARY')
print('=' * 90)
for _, row in summary.iterrows():
    print(f"\n{row['Risk']}")
    print(f"  Method:        {row['Quant Method']}")
    print(f"  Key Output:    {row['Key Output']}")
    print(f"  RT&RP Action:  {row['RT&RP Action']}")

print('\n\nAll analyses complete. Figures saved as fig1_r1_montecarlo.png, fig2_r2_sensitivity.png, fig3_r3_emv.png')